# STFT-based Accent Style Analysis Pipeline

이 노트북은 한 문장(`Interesting`)만 선택하여 Papago English / Korean-style / Japanese-style 발음을 비교하는 compact 버전입니다.

주요 구성:
1. 파일 로드
2. Waveform / FFT / Spectrogram을 `subplot axes`로 비교
3. Frame-level acoustic features 비교
4. MFCC 비교
5. 선택적 단어별 수동 분석
6. Korean-style → English-style 느낌의 간단한 변환 실험

나중에 직접 녹음한 음성으로 바꿀 때는 `DATA_DIR`와 파일명만 바꾸면 됩니다.


In [ ]:
# ==============================
# 0. Import libraries
# ==============================

import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

%matplotlib inline

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True


## 1. File loading

현재는 Papago 음성 파일을 사용합니다. 나중에 직접 녹음한 음성으로 바꿀 때는 `DATA_DIR`와 `target_files`만 수정하면 됩니다.

주의:
- 폴더명이 `PaPago m4a`처럼 공백이 있으면 그대로 써야 합니다.
- `Interesting_Papage_Korea.m4a` 파일명에 `Papage` 오타가 있으면 코드에서도 그대로 맞춰야 합니다.


In [ ]:
# ==============================
# 1. File path setting
# ==============================

DATA_DIR = "./PaPago m4a"   # 본인 폴더명에 맞게 수정

target_files = {
    "English": "Interesting_Papago_English.m4a",
    "Korea": "Interesting_Papage_Korea.m4a",
    "Japan": "Interesting_Papago_Japan.m4a",
}

sr_target = 22050

print("Current working directory:", os.getcwd())
print("Files in DATA_DIR:")
print(os.listdir(DATA_DIR))


In [ ]:
# ==============================
# 2. Load audio files
# ==============================

signals = {}

for accent, filename in target_files.items():
    path = os.path.join(DATA_DIR, filename)
    print("Loading:", path)

    y, sr = librosa.load(path, sr=sr_target)

    # Papago에서 다운로드한 음성은 일단 무음 제거하지 않음.
    # 직접 녹음한 음성에서 앞뒤 무음이 크면 아래 줄 주석 해제.
    # y, _ = librosa.effects.trim(y, top_db=30)

    signals[accent] = {
        "y": y,
        "sr": sr,
        "duration": len(y) / sr,
        "filename": filename
    }

for accent, data in signals.items():
    print(f"{accent}: {data['duration']:.3f} sec")


## 2. Basic comparison with subplots

각 발음 양식의 결과를 따로 출력하지 않고, 한 figure 안에 세 개의 axis로 묶어서 비교합니다.


In [ ]:
# ==============================
# 3. Waveform comparison
# ==============================

def plot_waveforms(signals):
    fig, axes = plt.subplots(len(signals), 1, figsize=(10, 5), sharex=False)

    if len(signals) == 1:
        axes = [axes]

    for ax, (accent, data) in zip(axes, signals.items()):
        y = data["y"]
        sr = data["sr"]

        librosa.display.waveshow(y, sr=sr, ax=ax)
        ax.set_title(f"Waveform - {accent} ({data['duration']:.2f} s)")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude")

    plt.tight_layout()
    plt.show()

plot_waveforms(signals)


In [ ]:
# ==============================
# 4. FFT spectrum comparison
# ==============================

def compute_fft(y, sr):
    N = len(y)
    Y = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(N, d=1/sr)
    magnitude = np.abs(Y)
    return freqs, magnitude

def plot_fft_comparison(signals, max_freq=8000):
    fig, axes = plt.subplots(len(signals), 1, figsize=(10, 6), sharex=True)

    if len(signals) == 1:
        axes = [axes]

    for ax, (accent, data) in zip(axes, signals.items()):
        y = data["y"]
        sr = data["sr"]

        freqs, mag = compute_fft(y, sr)

        ax.plot(freqs, mag)
        ax.set_title(f"FFT Magnitude Spectrum - {accent}")
        ax.set_ylabel("Magnitude")
        ax.set_xlim(0, max_freq)

    axes[-1].set_xlabel("Frequency (Hz)")
    plt.tight_layout()
    plt.show()

plot_fft_comparison(signals)


In [ ]:
# ==============================
# 5. Spectrogram comparison
# ==============================

n_fft = 1024
hop_length = 256

def plot_spectrograms(signals, max_freq=8000):
    fig, axes = plt.subplots(len(signals), 1, figsize=(10, 7), sharex=False)

    if len(signals) == 1:
        axes = [axes]

    last_img = None
    for ax, (accent, data) in zip(axes, signals.items()):
        y = data["y"]
        sr = data["sr"]

        D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, window="hann")
        S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

        last_img = librosa.display.specshow(
            S_db,
            sr=sr,
            hop_length=hop_length,
            x_axis="time",
            y_axis="hz",
            ax=ax
        )

        ax.set_ylim(0, max_freq)
        ax.set_title(f"STFT Spectrogram - {accent}")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")

    fig.colorbar(last_img, ax=axes, format="%+2.0f dB", shrink=0.75)
    plt.tight_layout()
    plt.show()

plot_spectrograms(signals)


## 3. Frame-level acoustic feature analysis

단어를 직접 자르지 않아도 전체 음성을 짧은 frame 단위로 분석할 수 있습니다.

비교할 feature:
- RMS energy: 시간별 에너지
- Spectral centroid: 주파수 중심. 고주파 자음이 강하면 커질 수 있음
- Spectral bandwidth: 주파수 분포 폭
- ZCR: zero crossing rate. 자음성/잡음성 구간에서 커질 수 있음
- Pitch contour: 억양 및 강세 변화


In [ ]:
# ==============================
# 6. Frame-level features
# ==============================

def frame_level_features(y, sr, hop_length=256, frame_length=1024):
    rms = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr,
        n_fft=frame_length,
        hop_length=hop_length
    )[0]

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr,
        n_fft=frame_length,
        hop_length=hop_length
    )[0]

    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    f0, voiced_flag, voiced_prob = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        frame_length=frame_length,
        hop_length=hop_length
    )

    times = librosa.frames_to_time(
        np.arange(len(rms)),
        sr=sr,
        hop_length=hop_length
    )

    return {
        "times": times,
        "rms": rms,
        "centroid": centroid,
        "bandwidth": bandwidth,
        "zcr": zcr,
        "pitch": f0,
        "voiced_flag": voiced_flag
    }


In [ ]:
# ==============================
# 7. Overlay comparison for one feature
# ==============================

def compare_overlay_feature(signals, feature_name, ylabel=None, ylim=None):
    plt.figure(figsize=(10, 3.5))

    for accent, data in signals.items():
        feat = frame_level_features(data["y"], data["sr"])
        times = feat["times"]
        values = feat[feature_name]

        plt.plot(times, values, label=accent, linewidth=1.5)

    plt.title(f"Frame-level {feature_name} comparison")
    plt.xlabel("Time (s)")
    plt.ylabel(ylabel if ylabel else feature_name)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.legend()
    plt.tight_layout()
    plt.show()

compare_overlay_feature(signals, "rms", ylabel="RMS energy")
compare_overlay_feature(signals, "centroid", ylabel="Spectral centroid (Hz)")
compare_overlay_feature(signals, "bandwidth", ylabel="Spectral bandwidth (Hz)")
compare_overlay_feature(signals, "zcr", ylabel="Zero crossing rate")
compare_overlay_feature(signals, "pitch", ylabel="Pitch / F0 (Hz)", ylim=(50, 400))


In [ ]:
# ==============================
# 8. Feature comparison in subplots
# ==============================

def plot_all_frame_features(signals):
    feature_specs = [
        ("rms", "RMS energy", None),
        ("centroid", "Spectral centroid (Hz)", None),
        ("bandwidth", "Spectral bandwidth (Hz)", None),
        ("zcr", "Zero crossing rate", None),
        ("pitch", "Pitch / F0 (Hz)", (50, 400)),
    ]

    fig, axes = plt.subplots(len(feature_specs), 1, figsize=(10, 9), sharex=False)

    for ax, (feature_name, ylabel, ylim) in zip(axes, feature_specs):
        for accent, data in signals.items():
            feat = frame_level_features(data["y"], data["sr"])
            ax.plot(feat["times"], feat[feature_name], label=accent, linewidth=1.2)

        ax.set_title(feature_name)
        ax.set_ylabel(ylabel)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.legend(loc="upper right")

    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

plot_all_frame_features(signals)


## 4. Summary feature table

전체 음성 단위의 duration, 평균 energy, pitch, spectral centroid 등을 표로 정리합니다.


In [ ]:
# ==============================
# 9. Extract summary features
# ==============================

def extract_summary_features(y, sr):
    feat = frame_level_features(y, sr)

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13,
        n_fft=n_fft,
        hop_length=hop_length
    )

    return {
        "duration": len(y) / sr,
        "mean_rms": np.nanmean(feat["rms"]),
        "std_rms": np.nanstd(feat["rms"]),
        "mean_centroid": np.nanmean(feat["centroid"]),
        "mean_bandwidth": np.nanmean(feat["bandwidth"]),
        "mean_zcr": np.nanmean(feat["zcr"]),
        "mean_pitch": np.nanmean(feat["pitch"]),
        "std_pitch": np.nanstd(feat["pitch"]),
        "voiced_ratio": np.nanmean(feat["voiced_flag"]),
        "mfcc_mean": np.nanmean(mfcc, axis=1)
    }

summary_rows = []

for accent, data in signals.items():
    summary = extract_summary_features(data["y"], data["sr"])

    row = {k: v for k, v in summary.items() if k != "mfcc_mean"}
    row["accent"] = accent
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary[[
    "accent",
    "duration",
    "mean_rms",
    "std_rms",
    "mean_centroid",
    "mean_bandwidth",
    "mean_zcr",
    "mean_pitch",
    "std_pitch",
    "voiced_ratio"
]]

df_summary


In [ ]:
# ==============================
# 10. Bar plots for summary features
# ==============================

def plot_summary_bars(df_summary):
    metrics = [
        "duration",
        "mean_rms",
        "mean_centroid",
        "mean_bandwidth",
        "mean_zcr",
        "mean_pitch",
        "std_pitch",
        "voiced_ratio"
    ]

    fig, axes = plt.subplots(4, 2, figsize=(10, 10))
    axes = axes.flatten()

    for ax, metric in zip(axes, metrics):
        ax.bar(df_summary["accent"], df_summary[metric])
        ax.set_title(metric)
        ax.set_xlabel("Accent")
        ax.set_ylabel(metric)

    plt.tight_layout()
    plt.show()

plot_summary_bars(df_summary)


## 5. MFCC analysis

MFCC는 음성의 spectral envelope를 요약하는 feature입니다. 여기서는 각 발음 양식의 MFCC heatmap과 평균 MFCC 거리 비교를 수행합니다.


In [ ]:
# ==============================
# 11. MFCC heatmaps in subplots
# ==============================

def plot_mfccs(signals):
    fig, axes = plt.subplots(len(signals), 1, figsize=(10, 7), sharex=False)

    if len(signals) == 1:
        axes = [axes]

    last_img = None
    for ax, (accent, data) in zip(axes, signals.items()):
        y = data["y"]
        sr = data["sr"]

        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=13,
            n_fft=n_fft,
            hop_length=hop_length
        )

        last_img = librosa.display.specshow(
            mfcc,
            sr=sr,
            hop_length=hop_length,
            x_axis="time",
            ax=ax
        )

        ax.set_title(f"MFCC - {accent}")
        ax.set_ylabel("MFCC coef.")

    fig.colorbar(last_img, ax=axes, shrink=0.75)
    plt.tight_layout()
    plt.show()

plot_mfccs(signals)


In [ ]:
# ==============================
# 12. MFCC distance comparison
# ==============================

mfcc_means = {}

for accent, data in signals.items():
    y = data["y"]
    sr = data["sr"]

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13,
        n_fft=n_fft,
        hop_length=hop_length
    )

    mfcc_means[accent] = np.mean(mfcc, axis=1)

distance_rows = []

accents = list(mfcc_means.keys())

for i in range(len(accents)):
    for j in range(i + 1, len(accents)):
        a = accents[i]
        b = accents[j]
        dist = np.linalg.norm(mfcc_means[a] - mfcc_means[b])

        distance_rows.append({
            "pair": f"{a} vs {b}",
            "mfcc_distance": dist
        })

df_mfcc_dist = pd.DataFrame(distance_rows)
df_mfcc_dist


In [ ]:
plt.figure(figsize=(7, 3.5))
plt.bar(df_mfcc_dist["pair"], df_mfcc_dist["mfcc_distance"])
plt.title("MFCC Mean Vector Distance")
plt.xlabel("Accent pair")
plt.ylabel("Euclidean distance")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 6. Optional word-level analysis

실제 적용에서는 단어를 매번 수동으로 자를 수 없으므로 frame-level 분석이 메인입니다. 다만 보고서에서 특정 단어 2~3개만 정밀 분석하고 싶을 때 아래 셀을 사용합니다.

아래 시간 구간은 예시입니다. Waveform을 보고 직접 수정해야 합니다.


In [ ]:
# ==============================
# 13. Optional word-level segmentation
# ==============================

def crop_by_time(y, sr, start, end):
    return y[int(start * sr):int(end * sr)]

# 아래 시간은 예시이므로 반드시 본인 그래프를 보고 수정하세요.
# 문장: I think this signal processing project is really interesting.

word_segments = {
    "English": {
        "think": (0.20, 0.55),
        "signal": (0.55, 1.05),
        "project": (1.65, 2.05),
        "interesting": (2.35, 3.05),
    },
    "Korea": {
        "think": (0.20, 0.65),
        "signal": (0.65, 1.20),
        "project": (1.90, 2.40),
        "interesting": (2.75, 3.60),
    },
    "Japan": {
        "think": (0.20, 0.70),
        "signal": (0.70, 1.30),
        "project": (2.00, 2.65),
        "interesting": (3.00, 3.90),
    }
}


In [ ]:
# ==============================
# 14. Word-level feature table
# ==============================

word_rows = []

for accent, segments in word_segments.items():
    y = signals[accent]["y"]
    sr = signals[accent]["sr"]

    for word, (start, end) in segments.items():
        y_word = crop_by_time(y, sr, start, end)
        summary = extract_summary_features(y_word, sr)

        word_rows.append({
            "accent": accent,
            "word": word,
            "start": start,
            "end": end,
            "duration": summary["duration"],
            "mean_rms": summary["mean_rms"],
            "mean_centroid": summary["mean_centroid"],
            "mean_bandwidth": summary["mean_bandwidth"],
            "mean_zcr": summary["mean_zcr"],
            "mean_pitch": summary["mean_pitch"],
            "std_pitch": summary["std_pitch"],
            "voiced_ratio": summary["voiced_ratio"]
        })

df_words = pd.DataFrame(word_rows)
df_words


In [ ]:
# ==============================
# 15. Word-level metric comparison
# ==============================

def plot_word_metric(df_words, metric):
    fig, ax = plt.subplots(figsize=(8, 3.5))

    for accent in df_words["accent"].unique():
        subset = df_words[df_words["accent"] == accent]
        ax.plot(subset["word"], subset[metric], marker="o", label=accent)

    ax.set_title(f"Word-level {metric} comparison")
    ax.set_xlabel("Word")
    ax.set_ylabel(metric)
    ax.legend()
    plt.tight_layout()
    plt.show()

for metric in ["duration", "mean_rms", "mean_centroid", "mean_pitch"]:
    plot_word_metric(df_words, metric)


In [ ]:
# ==============================
# 16. Word-level spectrograms as subplots
# ==============================

def plot_word_spectrograms(word):
    fig, axes = plt.subplots(len(signals), 1, figsize=(8, 5), sharex=False)

    if len(signals) == 1:
        axes = [axes]

    last_img = None
    for ax, accent in zip(axes, signals.keys()):
        y = signals[accent]["y"]
        sr = signals[accent]["sr"]

        start, end = word_segments[accent][word]
        y_word = crop_by_time(y, sr, start, end)

        D = librosa.stft(y_word, n_fft=n_fft, hop_length=hop_length)
        S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

        last_img = librosa.display.specshow(
            S_db,
            sr=sr,
            hop_length=hop_length,
            x_axis="time",
            y_axis="hz",
            ax=ax
        )

        ax.set_ylim(0, 8000)
        ax.set_title(f"{word} - {accent}")

    fig.colorbar(last_img, ax=axes, format="%+2.0f dB", shrink=0.75)
    plt.tight_layout()
    plt.show()

# 예시: 특정 단어만 비교
plot_word_spectrograms("think")
plot_word_spectrograms("project")


## 7. Simple Korean-style to English-style transformation

이 실험은 완전한 accent conversion이 아닙니다. 다만 Korean-style 발음을 English-style에 가깝게 만들기 위해 다음 신호처리 조작을 시도합니다.

1. Time stretch: 발화 길이 단축
2. High-frequency boost: 자음 선명도 증가
3. 변환 전후 waveform / spectrogram / feature 비교


In [ ]:
# ==============================
# 17. Simple transformation functions
# ==============================

def high_frequency_boost(y, sr, cutoff=3000, boost=1.5):
    Y = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(len(y), d=1/sr)

    gain = np.ones_like(freqs)
    gain[freqs >= cutoff] = boost

    Y_boosted = Y * gain
    y_boosted = np.fft.irfft(Y_boosted, n=len(y))

    max_abs = np.max(np.abs(y_boosted))
    if max_abs > 0:
        y_boosted = y_boosted / max_abs * 0.9

    return y_boosted

# Korean-style 음성을 English-style 길이에 가깝게 time stretch
y_korea = signals["Korea"]["y"]
sr = signals["Korea"]["sr"]

english_duration = signals["English"]["duration"]
korea_duration = signals["Korea"]["duration"]

# librosa time_stretch의 rate가 클수록 길이가 짧아짐.
stretch_rate = korea_duration / english_duration

print("English duration:", english_duration)
print("Korea duration:", korea_duration)
print("Stretch rate:", stretch_rate)

y_korea_fast = librosa.effects.time_stretch(y_korea, rate=stretch_rate)
y_korea_converted = high_frequency_boost(y_korea_fast, sr, cutoff=3000, boost=1.4)

signals_converted = {
    "English": signals["English"],
    "Korea_original": signals["Korea"],
    "Korea_converted": {
        "y": y_korea_converted,
        "sr": sr,
        "duration": len(y_korea_converted) / sr,
        "filename": "converted"
    }
}

for accent, data in signals_converted.items():
    print(f"{accent}: {data['duration']:.3f} sec")


In [ ]:
# ==============================
# 18. Converted speech comparison
# ==============================

plot_waveforms(signals_converted)
plot_spectrograms(signals_converted)
plot_all_frame_features(signals_converted)


In [ ]:
# ==============================
# 19. Save converted audio
# ==============================

import soundfile as sf

OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

converted_path = os.path.join(OUTPUT_DIR, "Interesting_Korea_converted.wav")
sf.write(converted_path, y_korea_converted, sr)

print("Saved:", converted_path)


## 8. Next step: replace Papago files with your own voice

최종 제출용으로는 직접 녹음한 음성 3개 이상이 필요합니다.

추천 녹음 방식:
1. 평소 한국식 영어 발음
2. 또박또박 읽은 영어 발음
3. Papago English를 듣고 따라 한 영어 발음

같은 문장:
`I think this signal processing project is really interesting.`

녹음 파일 이름만 아래처럼 맞추면 이 노트북 구조를 거의 그대로 사용할 수 있습니다.

- `Interesting_MyVoice_Normal.m4a`
- `Interesting_MyVoice_Clear.m4a`
- `Interesting_MyVoice_Imitation.m4a`
